### Importing Libraries

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
# Load the data
df = pd.read_csv("fau_transline_recruitment.csv")

In [3]:
# Drop irrelevant columns
df_with_relavant_columns = df.drop(columns=['depot_driver', 'city_bus_driver', 'distance_from_company', 'recruitment_strategy'])

### Simplifying the Dataset
To streamline the dataset and prepare it for the Apriori algorithm, we will apply binning on relevant columns to convert continuous values into categorical ranges. This reduces dimensionality and makes the data more suitable for association rule mining.

In [4]:
# Creating a copy of the dataframe
binned_df = df_with_relavant_columns.copy()

binned_df['experience_years'] = pd.cut(
    df_with_relavant_columns['experience_years'],
    bins=[0, 3, 7, 12, 15],
    labels=[
        'Novice[0–3]',
        'Skilled[3–7]',
        'Advanced[7–10]',
        'Expert[10-15]'
    ],
    right=True,
    include_lowest=True
)

binned_df['previous_companies'] = pd.cut(
    df_with_relavant_columns['previous_companies'],
    bins=[0, 1, 3, 5],
    labels=['[0-1]', '[2-3]', '[4-5]'],
    include_lowest=True
)

skill_columns = ['safe_driving_skills', 'customer_service_skills', 'navigation_skills']
skills_columns_bins = [0, 50, 70, 90, 100]
skills_columns_labels = ['Poor[0–60]', 'Avg[60–75]', 'Good[75–85]', 'Best[85–100]']

for col in skill_columns:
    binned_df[col] = pd.cut(
        df_with_relavant_columns[col],
        bins=skills_columns_bins,
        labels=skills_columns_labels,
        right=True,
        include_lowest=True
    )

### One-hot Encode the Categorical Columns
To prepare the binned categorical data for the Apriori algorithm, we apply one-hot encoding. This process converts each category into a separate binary column, allowing the algorithm to efficiently identify the presence or absence of each category within transactions.

In [5]:
# One-hot encode the categorical columns
df_encoded = pd.get_dummies(binned_df, columns=['education_level', 'experience_years', 'previous_companies', 'safe_driving_skills', 'customer_service_skills', 'navigation_skills'])

###  Applying Apriori Algorithm

In [6]:
# Apply apriori algorithm to find frequent itemsets
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

# Filter rules where the rules_filtered_antecedents are 'intercity_bus_driver', 'hiring_decision'
outcome_set = set(['intercity_bus_driver', 'hiring_decision'])

filtered_rules_antecedents = rules[rules['antecedents'].apply(lambda x: len(x.intersection(outcome_set)) == 0)]
filtered_rules_consequents = filtered_rules_antecedents[filtered_rules_antecedents['consequents'] == outcome_set]

# Sort rules by lift in descending order
sorted_rules = filtered_rules_consequents.sort_values(by=['lift'], ascending=False)

### Displaying the Top 10 Rules Sorted by Lift in Desc Order

In [7]:
for idx, rule in sorted_rules.head(10).iterrows():
    print(f"Rule {idx + 1}:")
    print(f"  Antecedents : {set(rule['antecedents'])}")
    print(f"  Support     : {rule['support']:.3f}")
    print(f"  Confidence  : {rule['confidence']:.3f}")
    print(f"  Lift        : {rule['lift']:.3f}")
    print("-" * 40)

Rule 3042:
  Antecedents : {'customer_service_skills_Good[75–85]', 'education_level_realschule_or_higher', 'safe_driving_skills_Good[75–85]'}
  Support     : 0.010
  Confidence  : 0.938
  Lift        : 3.111
----------------------------------------
Rule 747:
  Antecedents : {'education_level_hauptschule', 'safe_driving_skills_Good[75–85]'}
  Support     : 0.017
  Confidence  : 0.897
  Lift        : 2.975
----------------------------------------
Rule 2999:
  Antecedents : {'customer_service_skills_Good[75–85]', 'experience_years_Expert[10-15]', 'education_level_realschule_or_higher'}
  Support     : 0.010
  Confidence  : 0.882
  Lift        : 2.928
----------------------------------------
Rule 2979:
  Antecedents : {'experience_years_Advanced[7–10]', 'education_level_realschule_or_higher', 'safe_driving_skills_Good[75–85]'}
  Support     : 0.012
  Confidence  : 0.857
  Lift        : 2.845
----------------------------------------
Rule 2986:
  Antecedents : {'experience_years_Advanced[7–1